# SVM multi-label classifier for app reviews
Classifies each `review` into: `dependence`, `attachment`, `isolation_displacement`, `crisis_response_frustration`, `other_none`.

**Model:** TF-IDF (word unigrams) → one LinearSVC per label (class-balanced) with a tuned decision threshold.
Only the review text is used as input; `sample_theme`, `sample_band`, `ai_*` and `annotator_notes` are excluded because they would leak the answer.

## 1. Setup & load data

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, f1_score

warnings.filterwarnings("ignore")
SEED = 42

# >>> change this path if the CSV is somewhere else <<<
CSV = "annonated_training_test_set.csv"

df = pd.read_csv(CSV)
LABELS = [c for c in df.columns if c.startswith("label_")]
names = [c.replace("label_", "") for c in LABELS]

Y = (df[LABELS] == "Yes").astype(int).values   # NaN / "No" -> 0
X = df["review"].fillna("").astype(str).values

print(f"{len(df)} reviews")
pd.Series(Y.sum(axis=0), index=names, name="positives")

300 reviews


dependence                      14
attachment                      21
isolation_displacement          19
crisis_response_frustration     23
other_none                     233
Name: positives, dtype: int64

## 2. Train / test split
Stratified on each row's rarest label so every rare class appears in both splits.

In [2]:
freq = Y.sum(axis=0)
strat = np.array([min(np.where(r)[0], key=lambda j: freq[j]) for r in Y])

X_tr, X_te, Y_tr, Y_te, s_tr, _ = train_test_split(
    X, Y, strat, test_size=0.2, stratify=strat, random_state=SEED)
print(len(X_tr), "train /", len(X_te), "test")

240 train / 60 test


## 3. Model

In [3]:
class SVMReviewClassifier:
    """TF-IDF + one LinearSVC per label. A threshold slightly below 0
    trades precision for recall, which helps the rare labels."""
    def __init__(self, C=0.1, shift=-0.2):
        self.C, self.shift = C, shift

    def fit(self, X, Y):
        self.vec = TfidfVectorizer(sublinear_tf=True, strip_accents="unicode")
        A = self.vec.fit_transform(X)
        self.models = [
            LinearSVC(C=self.C, class_weight="balanced", random_state=SEED).fit(A, Y[:, j])
            for j in range(Y.shape[1])
        ]
        return self

    def decision_function(self, X):
        A = self.vec.transform(X)
        return np.column_stack([m.decision_function(A) for m in self.models])

    def predict(self, X):
        return (self.decision_function(X) > self.shift).astype(int)

## 4. Tune `C` and the threshold
5-fold CV (repeated with 3 seeds) on the **training split only**, scored by macro-F1.

In [4]:
shifts = [0, -0.1, -0.2, -0.3, -0.4]
Cs = [0.03, 0.1, 0.3, 1]

def cv_scores(C, Xs, Ys, ss, k=5, seeds=(0, 1, 2)):
    out = {sh: [] for sh in shifts}
    for sd in seeds:
        oof = np.zeros(Ys.shape)
        for tr, va in StratifiedKFold(k, shuffle=True, random_state=sd).split(Xs, ss):
            oof[va] = SVMReviewClassifier(C).fit(Xs[tr], Ys[tr]).decision_function(Xs[va])
        for sh in shifts:
            out[sh].append(f1_score(Ys, (oof > sh).astype(int), average="macro", zero_division=0))
    return {sh: np.mean(v) for sh, v in out.items()}

grid = pd.DataFrame({C: cv_scores(C, X_tr, Y_tr, s_tr) for C in Cs}).T
grid.index.name, grid.columns.name = "C", "threshold shift"
display(grid.round(3))

best_C = grid.max(axis=1).idxmax()
best_shift = grid.loc[best_C].idxmax()
print(f"Best: C={best_C}, shift={best_shift}  (macro-F1 {grid.loc[best_C, best_shift]:.3f})")

threshold shift,0.0,-0.1,-0.2,-0.3,-0.4
C,,,,,
0.03,0.353,0.394,0.302,0.275,0.273
0.10,0.325,0.405,0.415,0.358,0.321
0.30,0.306,0.364,0.417,0.411,0.384
1.00,0.305,0.358,0.370,0.421,0.400


Best: C=1.0, shift=-0.3  (macro-F1 0.421)


## 5. Evaluate

In [5]:
model = SVMReviewClassifier(best_C, best_shift).fit(X_tr, Y_tr)
pred = model.predict(X_te)

print("=== Hold-out test set (n=%d) ===" % len(X_te))
print(classification_report(Y_te, pred, target_names=names, zero_division=0, digits=3))

=== Hold-out test set (n=60) ===
                             precision    recall  f1-score   support

                 dependence      0.000     0.000     0.000         3
                 attachment      0.167     0.250     0.200         4
     isolation_displacement      0.000     0.000     0.000         3
crisis_response_frustration      0.600     0.750     0.667         4
                 other_none      0.825     1.000     0.904        47

                  micro avg      0.699     0.836     0.761        61
                  macro avg      0.318     0.400     0.354        61
               weighted avg      0.686     0.836     0.753        61
                samples avg      0.767     0.850     0.794        61



Since the test set has only 3–4 examples per rare label, the numbers above might be noisy

## 6. Inspect

In [6]:
final = SVMReviewClassifier(best_C, best_shift).fit(X, Y)

# checking for more influential words per label
vocab = np.array(final.vec.get_feature_names_out())
for j, m in enumerate(final.models):
    print(f"{names[j]:<30}", ", ".join(vocab[np.argsort(m.coef_.ravel())[-8:][::-1]]))

dependence                     always, actual, care, woebot, 10, step, attachments, icon
attachment                     friend, he, speaking, anytime, who, wants, woebot, always
isolation_displacement         anytime, gets, lonely, desperate, feeling, less, want, personal
crisis_response_frustration    worse, crisis, what, then, no, failed, suicidal, ll
other_none                     good, really, ai, helps, depression, whenever, anyone, app


Some of these words doesn't relate to the theme at all, which is probably why it's doing a poor job at classifying. Maybe we need more sample?